In [ ]:
(* load the setup of the notebooks *)
#use "../setup.ml";;

# How to Evaluate Propositional Formulas

In this section, we want to show how to evaluate formulas from <em>propositional logic</em>.  First of all we need to discuss how to represent logical formulas.  We represent formulas as values of the *variant type* `formula` that is defined in the notebook `Propositional-Logic-Parser.ipynb`:
```
type formula =
  | Var of string               (* propositional variable *)
  | Verum                       (* ⊤ *)
  | Falsum                      (* ⊥ *)
  | Not of formula              (* ¬ *)
  | And of formula * formula    (* ∧ *)
  | Or  of formula * formula    (* ∨ *)
  | Imp of formula * formula    (* → *)
  | Iff of formula * formula    (* ↔ *)
```
When we write formulas as strings, we represent the propositional operators as unicode symbols according to the following table:

<table style="border-collapse: collapse; margin-left: auto; margin-right: auto;">
    <tr>
        <th style="border: 2px solid blue; padding: 8px 15px; vertical-align: middle; text-align: center;">Colloquial Name</th>
        <th style="border-top: 2px solid blue; border-bottom: 2px solid blue; border-right: 2px solid blue; padding: 8px 15px; vertical-align: middle; text-align: center;">Mathematical Name</th>
        <th style="border-top: 2px solid blue; border-bottom: 2px solid blue; border-right: 2px solid blue; padding: 8px 15px; vertical-align: middle; text-align: center;">Symbol</th>
        <th style="border-top: 2px solid blue; border-bottom: 2px solid blue; border-right: 2px solid blue; padding: 8px 15px; vertical-align: middle; text-align: center;">Unicode</th>
    </tr>
    <tr>
        <td style="border-left: 2px solid blue; border-right: 2px solid blue; padding: 8px 15px; vertical-align: middle; text-align: center;"><tt>and</tt></td>
        <td style="border-right: 2px solid blue; padding: 8px 15px; vertical-align: middle; text-align: center;"><tt>conjunction</tt></td>
        <td style="font-size: 2em; border-right: 2px solid blue; padding: 8px 15px; vertical-align: middle; text-align: center;">∧</td>
        <td style="border-right: 2px solid blue; padding: 8px 15px; vertical-align: middle; text-align: center;">U+2227</td>
    </tr>
    <tr>
        <td style="border-left: 2px solid blue; border-right: 2px solid blue; padding: 8px 15px; vertical-align: middle; text-align: center;"><tt>or</tt></td>
        <td style="border-right: 2px solid blue; padding: 8px 15px; vertical-align: middle; text-align: center;"><tt>disjunction</tt></td>
        <td style="font-size: 2em; border-right: 2px solid blue; padding: 8px 15px; vertical-align: middle; text-align: center;">∨</td>
        <td style="border-right: 2px solid blue; padding: 8px 15px; vertical-align: middle; text-align: center;">U+2228</td>
    </tr>
    <tr>
        <td style="border-left: 2px solid blue; border-right: 2px solid blue; padding: 8px 15px; vertical-align: middle; text-align: center;"><tt>not</tt></td>
        <td style="border-right: 2px solid blue; padding: 8px 15px; vertical-align: middle; text-align: center;"><tt>negation</tt></td>
        <td style="font-size: 2em; border-right: 2px solid blue; padding: 8px 15px; vertical-align: middle; text-align: center;">¬</td>
        <td style="border-right: 2px solid blue; padding: 8px 15px; vertical-align: middle; text-align: center;">U+00ac</td>
    </tr>
    <tr>
        <td style="border-left: 2px solid blue; border-right: 2px solid blue; padding: 8px 15px; vertical-align: middle; text-align: center;"><tt>implies</tt></td>
        <td style="border-right: 2px solid blue; padding: 8px 15px; vertical-align: middle; text-align: center;"><tt>conditional</tt></td>
        <td style="font-size: 2em; border-right: 2px solid blue; padding: 8px 15px; vertical-align: middle; text-align: center;">→</td>
        <td style="border-right: 2px solid blue; padding: 8px 15px; vertical-align: middle; text-align: center;">U+2192</td>
    </tr>
    <tr>
        <td style="border-left: 2px solid blue; border-right: 2px solid blue; padding: 8px 15px; vertical-align: middle; text-align: center;"><tt>if and only if</tt></td>
        <td style="border-right: 2px solid blue; padding: 8px 15px; vertical-align: middle; text-align: center;"><tt>biconditional</tt></td>
        <td style="font-size: 2em; border-right: 2px solid blue; padding: 8px 15px; vertical-align: middle; text-align: center;">↔</td>
        <td style="border-right: 2px solid blue; padding: 8px 15px; vertical-align: middle; text-align: center;">U+2194</td>
    </tr>
    <tr>
        <td style="border-left: 2px solid blue; border-right: 2px solid blue; padding: 8px 15px; vertical-align: middle; text-align: center;"><tt></tt></td>
        <td style="border-right: 2px solid blue; padding: 8px 15px; vertical-align: middle; text-align: center;"><tt>falsum</tt></td>
        <td style="font-size: 2em; border-right: 2px solid blue; padding: 8px 15px; vertical-align: middle; text-align: center;">⊥</td>
        <td style="border-right: 2px solid blue; padding: 8px 15px; vertical-align: middle; text-align: center;">U+22a5</td>
    </tr>
    <tr>
        <td style="border-bottom: 2px solid blue; border-left: 2px solid blue; border-right: 2px solid blue; padding: 8px 15px; vertical-align: middle; text-align: center;"><tt></tt></td>
        <td style="border-bottom: 2px solid blue; border-right: 2px solid blue; padding: 8px 15px; vertical-align: middle; text-align: center;"><tt>verum</tt></td>
        <td style="font-size: 2em; border-bottom: 2px solid blue; border-right: 2px solid blue; padding: 8px 15px; vertical-align: middle; text-align: center;">⊤</td>
        <td style="border-bottom: 2px solid blue; border-right: 2px solid blue; padding: 8px 15px; vertical-align: middle; text-align: center;">U+22a4</td>
    </tr>
</table>

For example, the formula
$$ (p \rightarrow q) \wedge (\neg p \rightarrow q) \rightarrow q $$
will be represented as the following value of type `formula`:
```
Imp (And (Imp (Var "p", Var "q"), Imp (Not (Var "p"), Var "q")), Var "q")
```

In an OCaml string, we can write a Unicode character using the escape sequence `\u{X...X}`, where `X...X` are the hexadecimal digits of its code point.  The string is stored in the UTF-8 encoding.

In [ ]:
let and_symbol = "\u{2227}"

On a computer running MacOs you can enable *Unicode Hex Input*  in *Settings*.  If you choose this input method, you first have to select the Unicode keyboard.
Then you hold down the option key while you input the hex code of the Unicode character you want to type.

In [ ]:
"⊤"

In [ ]:
let p_and_q = "p \u{2227} q"

The escape sequence works for every Unicode code point up to `U+10FFFF`, including the code points outside of the *Basic Multilingual Plane*.

In [ ]:
let smiley = "\u{1F600}"

$\top$

It is also possible to just copy and paste these symbols.  That is what I usually do.
Another possibility is to use $\LaTeX$ to create these symbols.  We have
* `$\top$` is rendered as $\top$,
* `$\bot$` is rendered as $\bot$,
* `$\neg$` is rendered as $\neg$,
* `$\wedge$` is rendered as $\wedge$,
* `$\vee$` is rendered as $\vee$,
* `$\rightarrow$` is rendered as $\rightarrow$,
* `$\leftrightarrow$` is rendered as $\leftrightarrow$.

As it is quite tedious to type formulas as values of type `formula`, I have written a parser for propositional formulas, which we will import.
The parser will correctly parse a string that represents a formula from propositional logic.  However, it will not detect
certain syntax errors:
  * The parser assumes that the operator $\wedge$ binds stronger than the operator $\vee$.
    In our lecture, we assume that the operators $\wedge$ and $\vee$ have the same precedence.

    Therefore, our parser parses the string $p \vee q \wedge r$ as if it was the formula $p \vee (q \wedge r)$.

    According to the conventions of our lecture, this string should be rejected as syntactically invalid.

  * The parser assumes that the operator $\leftrightarrow$ is right *associative*.

    Therefore, it parses the string $p \leftrightarrow q \leftrightarrow r$ as if it was the formula $p \leftrightarrow (q \leftrightarrow r)$.

    According to the conventions of our lecture, this string should be rejected as syntactically invalid.

The function `import_notebook` is defined in the file `setup.ml`.  It executes all definitions of the given notebook silently, i.e. without printing their types.  Tests and examples of the imported notebook are skipped.

In [ ]:
import_notebook "Propositional-Logic-Parser.ipynb"

The parser provides the function `parse` that takes a string `s` and returns the formula described by the string `s`.  Let us try to parse a complex formula.

In [ ]:
let s = "(p → q) ∧ (¬p → q) → q"
let f = parse s

Next, we need to decide how to represent a *propositional valuation*.  Mathematically, a *propositional valuation* $\mathcal{I}$ is a function of the form
$$ \mathcal{I}:\mathcal{P} \rightarrow \mathbb{B}  $$
that maps propositional variables to the set of Boolean values
$\mathbb{B} = \{\texttt{true}, \texttt{false}\}$.  We can represent a propositional  valuation $\mathcal{I}$ as the set of those variables that are evaluated as <tt>true</tt> by $\mathcal{I}$:
$$ \texttt{repr}(\mathcal{I}) := \bigl\{ v \in \mathcal{P} \mid \mathcal{I}(v) = \texttt{true} \bigr\}$$
Hence, in OCaml a propositional valuation has the type `string RSet.t`.  The module `RSet`, which is loaded by `setup.ml`, implements sets that can be nested.
For example, the following set would be a propositional valuation for the formula $\texttt{f}$ that interprets both `p` and `q` as `true`.  In OCaml, the names of variables have to start with a lower case letter.  Therefore, we write `i1` instead of $\mathcal{I}_1$.

In [ ]:
let i1 = RSet.of_list ["p"; "q"]

There are 4 possible propositional valuation for the formula $\texttt{f}$.  The other three propositional valuations are as follows:

In [ ]:
let i2 = RSet.of_list ["p"]
let i3 = RSet.of_list ["q"]
let i4 : string RSet.t = RSet.empty

In OCaml, the Boolean values are ordered: `false` is less than `true`.  Therefore, the implication $a \rightarrow b$ is true if and only if $a \leq b$.

In [ ]:
false < true

The function `evaluate` defined below takes two arguments:
   - `f` is a *propositional formula*.
   - `i` is a set of *propositional variables* representing a *propositional valuation*.

The function returns the value of the propositional formula `f` given the propositional valuation `i`.

**Implementation:** The function is defined by *pattern matching*: For every constructor of the type `formula` there is a case that specifies how a formula of this form is evaluated.  The OCaml compiler checks that the patterns cover all constructors.  If we forgot a case, the compiler would issue a warning.
- A variable `p` is true if and only if it is an element of the set `i`.
- When evaluating a formula of the form $g \rightarrow h$, the function uses the fact that the implication is true iff the value of $g$ is less than or equal to the value of $h$.
- A formula of the form $g \leftrightarrow h$ is true iff $g$ and $h$ have the same value.

In [ ]:
let rec evaluate (f : formula) (i : string RSet.t) : bool =
  match f with
  | Var p      -> RSet.mem p i
  | Verum      -> true
  | Falsum     -> false
  | Not g      -> not (evaluate g i)
  | And (g, h) -> evaluate g i && evaluate h i
  | Or  (g, h) -> evaluate g i || evaluate h i
  | Imp (g, h) -> evaluate g i <= evaluate h i
  | Iff (g, h) -> evaluate g i =  evaluate h i

Let's test this function for the formula
$$ f := (p \rightarrow q) \wedge (\neg p \rightarrow q) \rightarrow q. $$

In [ ]:
f

In [ ]:
List.iter
  (fun i -> Printf.printf "evaluate (%s) %s = %b\n" s (RSet.to_string Fun.id i) (evaluate f i))
  [i1; i2; i3; i4]

As the formula <tt>f</tt> is true for **every** possible propositional valuation `i`, it is a *tautology*.